# AllSortsHub Cartoon Studio — Wan 2.2 Episode 1 Colab Generator

This notebook generates Episode 1 on a Colab GPU and checkpoints completed clips to Google Drive.

**T4:** use the hardware-adaptive Wan 2.2 TI2V-5B quantized path with conservative settings designed to reduce memory pressure. **24GB+ GPUs:** use the official Wan 2.2 I2V-A14B path.

Do not delete the Google Drive checkpoint folder if Colab disconnects.

In [ ]:
# 1. GPU check — run this first
!nvidia-smi
import torch, shutil
print('PyTorch:', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU. In Colab choose Runtime > Change runtime type > GPU.')
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print('GPU:', GPU_NAME)
print('VRAM:', round(VRAM_GB, 1), 'GB')
print('FFmpeg:', shutil.which('ffmpeg'))
if VRAM_GB < 8: raise RuntimeError('At least 8 GB VRAM is required.')
USE_T4_PATH = VRAM_GB < 20
print('Generation path:', 'Wan 2.2 TI2V-5B T4-safe quantized' if USE_T4_PATH else 'Wan 2.2 I2V-A14B')

In [ ]:
# 2. Persistent Google Drive storage
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/AllSortsHub-Wan2.2'
!mkdir -p "$BASE/models" "$BASE/generated" "$BASE/output"
print(BASE)

In [ ]:
# 3. Clone project and install the Colab-safe backend
%cd /content
!rm -rf cartoon-studio
!git clone -q https://github.com/parth01/AllSortsHub-Cartoon-Studio.git cartoon-studio
!python -m pip install -q 'setuptools<82' wheel
!python -m pip install -q 'git+https://github.com/mkamranr/videogen.git'
!apt-get update -qq && apt-get install -y -qq ffmpeg
!videogen doctor --mode i2v || true

In [ ]:
# 4. Select and validate the exact model plan
import subprocess
if USE_T4_PATH:
    MODEL_ID = 'wan22'
    VARIANT = 'q8_0'
    PRESET = 'draft'
    T4_WIDTH, T4_HEIGHT, T4_FRAMES, T4_STEPS = 512, 288, 33, 4
    print(f'Using Wan 2.2 TI2V-5B q8_0 T4-safe mode: {T4_WIDTH}x{T4_HEIGHT}, {T4_FRAMES} frames, {T4_STEPS} steps')
    r = subprocess.run(['videogen','plan','AllSortsHub test','--mode','i2v','--model',MODEL_ID,'--variant',VARIANT,'--preset',PRESET,'--width',str(T4_WIDTH),'--height',str(T4_HEIGHT),'--frames',str(T4_FRAMES),'--steps',str(T4_STEPS),'--allow-slow'])
    if r.returncode != 0: raise RuntimeError('The T4 Wan 2.2 plan was refused. Do not download a large model.')
else:
    MODEL_ID = 'Wan-AI/Wan2.2-I2V-A14B'
    VARIANT = None
    PRESET = 'balanced'
    print('Using official Wan 2.2 I2V-A14B for 24GB+ GPU')

In [ ]:
# 5. Download the appropriate model only after the plan succeeds
import os, subprocess
if USE_T4_PATH:
    subprocess.run(['videogen','models','pull','wan22'], check=True)
else:
    from modelscope import snapshot_download
    A14B_DIR = os.path.join(BASE, 'models', 'Wan2.2-I2V-A14B')
    os.makedirs(A14B_DIR, exist_ok=True)
    snapshot_download('Wan-AI/Wan2.2-I2V-A14B', local_dir=A14B_DIR)
print('Model preparation complete.')

In [ ]:
# 6. Prepare Episode 1 and restore completed clips
from pathlib import Path
import json, shutil
ROOT = Path('/content/cartoon-studio/master-version/AllSortsHub-Billion 2')
LOCAL_GEN = ROOT / 'wan_i2v' / 'generated'
DRIVE_GEN = Path(BASE) / 'generated'
LOCAL_GEN.mkdir(parents=True, exist_ok=True)
DRIVE_GEN.mkdir(parents=True, exist_ok=True)
with open(ROOT / 'wan_i2v' / 'manifest.json') as f: manifest = json.load(f)
for p in DRIVE_GEN.glob('shot_*.mp4'):
    t = LOCAL_GEN / p.name
    if not t.exists() or t.stat().st_size < 10000: shutil.copy2(p, t)
print('Episode shots:', len(manifest['shots']))
print('Restored clips:', len(list(LOCAL_GEN.glob('shot_*.mp4'))))

In [ ]:
# 7. Resumable Episode 1 generation
import subprocess, shutil, sys, gc, time
prompts = (ROOT / 'wan_i2v' / 'prompts.txt').read_text()
STYLE = 'Modern 2D cel-shaded cartoon animation, bold clean black linework, semi-flat shading, vibrant colors, expressive facial acting, preserve the exact character designs and environment in the input image. Smooth readable hand-drawn motion. Keep faces, hair, clothing, proportions, props and background layout consistent.'
NEG = 'No photorealism, no 3D CGI, no live action, no extra fingers, no duplicate limbs, no warped faces, no character morphing, no costume changes, no hairstyle changes, no background replacement, no random objects, no random text, no logos, no watermark, no scene cuts, no extreme deformation.'
def prompt_for(n):
    marker = f'SHOT {n:02d} —'; start = prompts.find(marker)
    if start < 0: raise RuntimeError('Missing prompt for ' + marker)
    end = prompts.find('\n\nSHOT ', start + 2)
    if end < 0: end = prompts.find('\n\nNEGATIVE', start + 2)
    return f'{STYLE} {prompts[start:end if end >= 0 else None].split(chr(10),1)[1].strip()} {NEG}'
def copy_drive(p): shutil.copy2(p, DRIVE_GEN / p.name)
def cleanup_gpu():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available(): torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    except Exception: pass
def run_t4(image, prompt, out, seed):
    if out.exists() and out.stat().st_size > 10000: print('SKIP', out.name); return
    cmd=['videogen','i2v',str(image),prompt,'--model','wan22','--variant','q8_0','--preset','draft','--width',str(T4_WIDTH),'--height',str(T4_HEIGHT),'--frames',str(T4_FRAMES),'--steps',str(T4_STEPS),'--seed',str(seed),'--output',str(out),'--allow-slow','--no-plan']
    print('GENERATING', out.name, f'({T4_WIDTH}x{T4_HEIGHT}, {T4_FRAMES} frames, {T4_STEPS} steps)')
    cleanup_gpu()
    subprocess.run(cmd, check=True)
    cleanup_gpu()
def run_a14b(image, prompt, out, seed):
    if out.exists() and out.stat().st_size > 10000: print('SKIP', out.name); return
    WAN=Path('/content/Wan2.2')
    if not WAN.exists(): subprocess.run(['git','clone','-q','https://github.com/Wan-Video/Wan2.2.git',str(WAN)],check=True)
    cmd=[sys.executable,str(WAN/'generate.py'),'--task','i2v-A14B','--size','832*480','--ckpt_dir',str(A14B_DIR),'--offload_model','True','--convert_model_dtype','--t5_cpu','--frame_num','81','--image',str(image),'--prompt',prompt,'--base_seed',str(seed),'--save_file',str(out)]
    subprocess.run(cmd,cwd=WAN,check=True)
for shot in manifest['shots']:
    n=int(shot['id']); image=ROOT/shot['image']; duration=float(shot['duration']); p=prompt_for(n); out=LOCAL_GEN/f'shot_{n:02d}.mp4'
    if USE_T4_PATH:
        # T4-safe mode: render short chunks and assemble longer shots.
        if duration <= 3.0: run_t4(image,p,out,910000+n)
        else:
            a=LOCAL_GEN/f'shot_{n:02d}a.mp4'; b=LOCAL_GEN/f'shot_{n:02d}b.mp4'
            run_t4(image,p,a,910000+n)
            if not b.exists() or b.stat().st_size < 10000:
                frame=LOCAL_GEN/f'shot_{n:02d}_continuation.jpg'
                subprocess.run(['ffmpeg','-y','-sseof','-0.08','-i',str(a),'-frames:v','1','-q:v','2',str(frame)],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
                run_t4(frame,f'{STYLE} Continue exactly from this final frame. {p} {NEG}',b,1010000+n); frame.unlink(missing_ok=True)
            subprocess.run(['ffmpeg','-y','-i',str(a),'-i',str(b),'-filter_complex','[0:v][1:v]concat=n=2:v=1:a=0[v]','-map','[v]','-c:v','libx264','-pix_fmt','yuv420p',str(out)],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
    else: run_a14b(image,p,out,910000+n)
    if out.exists() and out.stat().st_size > 10000: copy_drive(out); print('CHECKPOINT',out.name)
    cleanup_gpu()
print('Generation pass complete.')

In [ ]:
# 8. Assemble final Episode 1 videos
%cd /content/cartoon-studio/master-version/AllSortsHub-Billion 2
!python3 wan_i2v/assemble_episode.py
!cp -f output/AllSortsHub_Episode_01_WAN_MASTER.mp4 "$BASE/output/"
!cp -f output/AllSortsHub_Episode_01_WAN_VERTICAL_9x16.mp4 "$BASE/output/"
!ls -lh output/AllSortsHub_Episode_01_WAN_MASTER.mp4 output/AllSortsHub_Episode_01_WAN_VERTICAL_9x16.mp4

## If Colab disconnects

Reconnect to a GPU, rerun the notebook setup cells, and rerun the generation cell. Completed MP4s in `MyDrive/AllSortsHub-Wan2.2/generated/` are restored and skipped.

**T4 note:** generation is intentionally conservative: 512x288, 33 frames, 4 steps. Do not switch to the 14B model on a T4.